# Prep tasks (schema version 4): what did the model find?

The same 20 recipes as the thinking comparison went through Claude Haiku 5.5 with the version-4 rules. Each recipe's answer now has a `state` (raw or cooked) per ingredient and a list of **prep tasks**: the knife work or marinating done before cooking. **The question:** are the tasks real, and are there enough of them to make cross-recipe prep (dice the onion once for two recipes) work?

A task is **stated** when a step names the technique ("dice the onion") and **implied** (`stated = False`) when a step only uses the ingredient in a way that needs it cut ("saute onion and garlic"). Implied tasks use a default technique (garlic: mince, onion: dice, herbs: chop) and never carry a size.

**How to review (about 20 minutes):**
1. Run all cells once. No API calls are made (needs only pandas).
2. For each recipe, read the numbered steps, then the prep-task table. Each task row shows the text of the step it points at.
3. In the small cell under the recipe, fill one verdict per task row, in order: `real` (the task is right), `invented` (nothing in the text supports it) or `wrong` (supported, but wrong technique, size, inputs or step). Use the `missing` note for prep the steps need that we did not return.
4. Set `REVIEWER` in the first code cell below, run the last cell, and it saves `review_prep_<name>.csv`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd()
assert (HERE / "stage1_rows.json").exists(), "open this notebook from pipelines/foodcom/pilot/"

REVIEWER = ""  # your first name, lowercase: the review is saved as review_prep_<name>.csv

stage1 = {r["source_ref"]: r for r in json.loads((HERE / "stage1_rows.json").read_text())}
order = list(stage1)


def load(filename):
    lines = (HERE / filename).read_text().splitlines()
    return {r["source_ref"]: r for r in map(json.loads, lines)}


runs = {
    "v3 (no prep tasks)": load("results_thinking_v3.jsonl"),
    "v4 strict (stated only)": load("results_thinking_v4_strict.jsonl"),
    "v4 (stated + implied)": load("results_thinking.jsonl"),
}
current = runs["v4 (stated + implied)"]
PRICE_IN, PRICE_OUT = 0.10, 0.50  # USD per million tokens, same as augment.py
FULL_RUN = 212_846
SIZED = {"dice", "chop", "slice", "julienne", "grate", "shred"}
verdicts, missing = {}, {}


def cost(rec):
    return (rec["input_tokens"] * PRICE_IN + rec["output_tokens"] * PRICE_OUT) / 1_000_000


def names_of(key):
    """Position -> ingredient name: the listed names, then the extras in the order the model gave them."""
    extras = [e["name"] for e in current[key]["result"]["extra_ingredients"]]
    return stage1[key]["ingredient_parts"] + extras


def form_of(task, names):
    """The Neo4j `form` we would build later in dbt: a size, or the sorted input names joined by ' + '."""
    if task["technique"] in {"marinate", "rub", "brine"}:
        return " + ".join(sorted(names[p].lower().strip() for p in task["with"]))
    return task["cut_size"] or ""


def all_tasks(run):
    return [t for rec in run.values() for t in rec["result"].get("prep_tasks", [])]

## At a glance

In [2]:
table = []
for label, run in runs.items():
    recs = list(run.values())
    tasks = all_tasks(run)
    table.append({
        "run": label,
        "valid": f"{sum(r['status'] == 'ok' for r in recs)}/20",
        "retries": sum(r["attempts"] > 1 for r in recs),
        "tokens in / recipe": round(sum(r["input_tokens"] for r in recs) / 20),
        "tokens out / recipe": round(sum(r["output_tokens"] for r in recs) / 20),
        "cost of the 20": f"${sum(cost(r) for r in recs):.4f}",
        "full run, direct": f"${sum(cost(r) for r in recs) / 20 * FULL_RUN:,.0f}",
        "full run, Batches": f"${sum(cost(r) for r in recs) / 20 * FULL_RUN / 2:,.0f}",
        "recipes with a task": sum(bool(r["result"].get("prep_tasks")) for r in recs),
        "tasks": len(tasks),
    })
display(pd.DataFrame(table).set_index("run"))

,valid,retries,tokens in / recipe,tokens out / recipe,cost of the 20,"full run, direct","full run, Batches",recipes with a task,tasks
run,,,,,,,,,
v3 (no prep tasks),20/20,0,2810,894,$0.0146,$155,$77,0,0
v4 strict (stated only),20/20,0,4078,1183,$0.0200,$213,$106,4,5
v4 (stated + implied),20/20,0,4460,1506,$0.0240,$255,$128,18,50


In [3]:
tasks = all_tasks(current)
per_recipe = pd.Series([len(r["result"]["prep_tasks"]) for r in current.values()])
ingredients = [i for r in current.values() for k in ("ingredients", "extra_ingredients") for i in r["result"][k]]

display(Markdown(f"**Tasks per recipe (v4):** {dict(per_recipe.value_counts().sort_index())} (tasks: recipes with that many)"))
display(pd.DataFrame({
    "technique": pd.Series([t["technique"] for t in tasks]).value_counts(),
}).T)
display(pd.DataFrame([{
    "stated": sum(t["stated"] for t in tasks),
    "implied": sum(not t["stated"] for t in tasks),
    "make_ahead true": sum(t["make_ahead"] for t in tasks),
    "make_ahead false": sum(not t["make_ahead"] for t in tasks),
    "has a cut_size": sum(t["cut_size"] is not None for t in tasks),
    "ingredients raw": sum(i["state"] == "raw" for i in ingredients),
    "ingredients cooked": sum(i["state"] == "cooked" for i in ingredients),
}], index=["v4"]))

**Tasks per recipe (v4):** {0: np.int64(2), 1: np.int64(5), 2: np.int64(5), 3: np.int64(4), 4: np.int64(2), 5: np.int64(1), 10: np.int64(1)} (tasks: recipes with that many)

,dice,chop,mince,slice,marinate,rub
technique,19,12,11,3,3,2


,stated,implied,make_ahead true,make_ahead false,has a cut_size,ingredients raw,ingredients cooked
v4,9,41,37,13,2,49,144


## Review the 20 recipes

In [4]:
def fmt_qty(i):
    return "to taste" if i["quantity"] is None else f"{i['quantity']:g} {i['unit']}"


def card(n, key):
    s, rec = stage1[key], current[key]
    names, answer = names_of(key), rec["result"]
    steps = s["instructions"]
    step_list = "\n".join(f"{k}. {t}" for k, t in enumerate(steps, 1))
    display(Markdown(f"### {n}. {s['name']}  \n{len(s['ingredient_parts'])} ingredients, {len(steps)} steps\n\n**Steps**\n\n{step_list}"))

    n_listed = len(s["ingredient_parts"])
    entries = [(i["position"], i, "no") for i in answer["ingredients"]]
    entries += [(n_listed + k, e, "yes") for k, e in enumerate(answer["extra_ingredients"])]
    display(pd.DataFrame([
        {"pos": pos, "ingredient": names[pos], "amount": fmt_qty(e), "state": e["state"], "extra": extra}
        for pos, e, extra in entries
    ]).style.hide(axis="index"))

    strict = len(runs["v4 strict (stated only)"][key]["result"]["prep_tasks"])
    pt = answer["prep_tasks"]
    if not pt:
        display(Markdown(f"**Prep tasks:** none (the strict run found {strict})"))
        return
    display(Markdown(f"**Prep tasks** ({len(pt)}; the strict run found {strict})"))
    display(pd.DataFrame([
        {
            "task": k,
            "ingredient": names[t["ingredient"]],
            "technique": t["technique"],
            "cut_size": t["cut_size"] or "",
            "with": " + ".join(names[p] for p in t["with"]),
            "make_ahead": t["make_ahead"],
            "stated": t["stated"],
            "step": t["source_step"],
            "text of that step": steps[t["source_step"] - 1],
        }
        for k, t in enumerate(pt, 1)
    ]).style.hide(axis="index").set_properties(**{"text-align": "left"}))

In [5]:
card(1, order[0])

### 1. Lemon Cucumbers  
8 ingredients, 5 steps

**Steps**

1. Place cucumbers and onion in a bowl.
2. Combine rest of the ingredients in a measuring cup.
3. Stir well.
4. Pour over cucumbers and onions.
5. Chill for 4 hours.

pos,ingredient,amount,state,extra
0,cucumbers,2 piece,raw,no
1,yellow onion,0.5 piece,raw,no
2,vinegar,0.25 cup,raw,no
3,sugar,2 tbsp,raw,no
4,bottled lemon juice,1 tbsp,raw,no
5,celery seed,1 tsp,raw,no
6,salt,0.75 tsp,raw,no
7,black pepper,0.125 tsp,raw,no


**Prep tasks:** none (the strict run found 0)

In [6]:
verdicts[1] = []  # one per task row, in order: real | invented | wrong
missing[1] = ""  # prep the steps need that we did not return

In [7]:
card(2, order[1])

### 2. Sour Pickle and Sweet Peppers Coleslaw  
7 ingredients, 1 steps

**Steps**

1. In large bowl, combine all ingredients very well. Season to taste with salt and pepper. Keep refrigerated.

pos,ingredient,amount,state,extra
0,red cabbage,0.5 piece,raw,no
1,red bell peppers,2 piece,raw,no
2,onion,0.5 piece,raw,no
3,dill pickle,1 piece,raw,no
4,fat-free mayonnaise,1 cup,raw,no
5,apple cider vinegar,1 cup,raw,no
6,celery seed,0.5 tsp,raw,no
7,black pepper,to taste,raw,yes


**Prep tasks** (3; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,red cabbage,dice,,,True,False,1,"In large bowl, combine all ingredients very well. Season to taste with salt and pepper. Keep refrigerated."
2,red bell peppers,dice,,,True,False,1,"In large bowl, combine all ingredients very well. Season to taste with salt and pepper. Keep refrigerated."
3,onion,dice,,,True,False,1,"In large bowl, combine all ingredients very well. Season to taste with salt and pepper. Keep refrigerated."


In [8]:
verdicts[2] = ["", "", ""]  # one per task row, in order: real | invented | wrong
missing[2] = ""  # prep the steps need that we did not return

In [9]:
card(3, order[2])

### 3. Mashed Potatoes With Sour Cream  
5 ingredients, 7 steps

**Steps**

1. Place potatoes in a heavy saucepan.
2. Cover with salted water.
3. Bring to a boil over medium high heat. Reduce heat to medium low.
4. Cover saucepan and simmer 15 minutes or until potatoes are tender. Drain, reserving 1/3 cup liquid.
5. Add butter, sour cream, and salt and pepper to taste.
6. Mash until smooth with an electric mixer or hand masher, adding reserved cooking liquid as necessary to moisten potatoes.
7. Stir in parsley. Serve potatoes with mushroom gravy.

pos,ingredient,amount,state,extra
0,russet potato,1 piece,cooked,no
1,unsalted butter,2 tbsp,cooked,no
2,unsalted butter,2 tbsp,cooked,no
3,sour cream,0.33 cup,raw,no
4,parsley,2 tbsp,raw,no
5,pepper,to taste,cooked,yes


**Prep tasks** (1; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,parsley,chop,,,False,False,7,Stir in parsley. Serve potatoes with mushroom gravy.


In [10]:
verdicts[3] = [""]  # one per task row, in order: real | invented | wrong
missing[3] = ""  # prep the steps need that we did not return

In [11]:
card(4, order[3])

### 4. Mama's Lasagna  
8 ingredients, 7 steps

**Steps**

1. Brown ground beef in a frying pan.
2. Add sauce and tomato paste.
3. In a bowl mix cottage cheese with egg, Parmesan cheese, and parsley.
4. Cover the bottom of a 9x13 inch pan with a little of the sauce.
5. Layer the uncooked noodles, then cottage cheese mixture, then meat sauce, then slices of mozzarella. Do one more layer in the same order.
6. Sprinkle top with Parmesan cheese.
7. Cover with foil and bake at 350°F for 45 minutes to an hour to make sure noodles are done.

pos,ingredient,amount,state,extra
0,ground beef,1 lb,cooked,no
1,tomato sauce,1 can,cooked,no
2,tomato paste,2 can,cooked,no
3,cottage cheese,1 lb,cooked,no
4,ricotta cheese,1 lb,cooked,no
5,egg,0.5 piece,cooked,no
6,fresh parsley,to taste,cooked,no
7,mozzarella cheese,1 lb,cooked,no
8,lasagna noodles,12 oz,cooked,yes
9,Parmesan cheese,to taste,cooked,yes


**Prep tasks** (2; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,fresh parsley,chop,,,False,False,3,"In a bowl mix cottage cheese with egg, Parmesan cheese, and parsley."
2,mozzarella cheese,slice,,,True,True,5,"Layer the uncooked noodles, then cottage cheese mixture, then meat sauce, then slices of mozzarella. Do one more layer in the same order."


In [12]:
verdicts[4] = ["", ""]  # one per task row, in order: real | invented | wrong
missing[4] = ""  # prep the steps need that we did not return

In [13]:
card(5, order[4])

### 5. Mom's Classic Tuna Melt  
4 ingredients, 5 steps

**Steps**

1. Preheat your oven on low broil.
2. Drain the water out of the tuna and combine the tuna and mayo in a bowl.
3. Cut the bagel in half to form two donuts and toast in a toaster.
4. Spread tuna on each half of the bagel and top with a slice of Swiss cheese.
5. Place in the oven until the cheese is melted.

pos,ingredient,amount,state,extra
0,tuna in water,1 can,cooked,no
1,mayonnaise,3 tbsp,cooked,no
2,swiss cheese,2 slice,cooked,no
3,bagel,1 piece,cooked,no


**Prep tasks:** none (the strict run found 0)

In [14]:
verdicts[5] = []  # one per task row, in order: real | invented | wrong
missing[5] = ""  # prep the steps need that we did not return

In [15]:
card(6, order[5])

### 6. Fava Beans Just Like Grandma Made  
8 ingredients, 7 steps

**Steps**

1. Shell the beans, add to boiling water, cook for apprx 10 minutes, dash into cold water drain.
2. Heat the oil and butter in a skillet.  Add onion and garlic and saute over medium heat until the onions are golden brown apprx 20 minutes.
3. Add Fava beans, tomato sauce, and pepper to the onions.
4. Place beans and onions in an oven proof dish and sprinkle with cheese.
5. To  make this dish ahead stop here and refrigerate.
6. Bake in 350 F oven for 20 minutes if beans are hot off the stove and 30 minutes if they have been in the frig.
7. Cheese will be melted, if you would like it browned will need to place under the broiled for a minute or two.

pos,ingredient,amount,state,extra
0,fava beans,2 lb,cooked,no
1,onion,1 piece,cooked,no
2,butter,1 tbsp,cooked,no
3,olive oil,1 tbsp,cooked,no
4,garlic clove,1 clove,cooked,no
5,tomato sauce,8 fl_oz,cooked,no
6,pepper,0.125 tsp,cooked,no
7,parmesan cheese,1 oz,cooked,no


**Prep tasks** (2; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,onion,dice,,,True,False,2,Heat the oil and butter in a skillet. Add onion and garlic and saute over medium heat until the onions are golden brown apprx 20 minutes.
2,garlic clove,mince,,,True,False,2,Heat the oil and butter in a skillet. Add onion and garlic and saute over medium heat until the onions are golden brown apprx 20 minutes.


In [16]:
verdicts[6] = ["", ""]  # one per task row, in order: real | invented | wrong
missing[6] = ""  # prep the steps need that we did not return

In [17]:
card(7, order[6])

### 7. Spicy Carrot Coins  
10 ingredients, 3 steps

**Steps**

1. In nonstick saucepan combine the first 8 ingredients. Bring to a boil. Reduce heat; cover and simmer for 5 minutes.
2. Discard cinnamon stick.
3. Add honey and lemon juice to carrots. Bring to a boil; cook, uncovered, for 5 to 8 minutes or until carrots are tender and liquid has evaporated.

pos,ingredient,amount,state,extra
0,carrots,2 lb,cooked,no
1,water,1.25 cup,cooked,no
2,cinnamon stick,1 piece,cooked,no
3,ground ginger,1 tsp,cooked,no
4,ground cumin,1 tsp,cooked,no
5,ground coriander,0.5 tsp,cooked,no
6,salt,0.25 tsp,cooked,no
7,cayenne pepper,0.125 tsp,cooked,no
8,honey,0.25 cup,cooked,no
9,lemon juice,4 tbsp,cooked,no


**Prep tasks** (1; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,carrots,dice,,,True,False,1,In nonstick saucepan combine the first 8 ingredients. Bring to a boil. Reduce heat; cover and simmer for 5 minutes.


In [18]:
verdicts[7] = [""]  # one per task row, in order: real | invented | wrong
missing[7] = ""  # prep the steps need that we did not return

In [19]:
card(8, order[7])

### 8. Black Bean Chile  
9 ingredients, 6 steps

**Steps**

1. Saute onion and garlic in olive oil.
2. Add jalapeno pepper, if desired; this chile is spicy enough for most people without the jalapeno.
3. Combine all ingredients, except the cilantro, in a pot or crockpot.
4. Cook on low heat until the chile is warmed through.
5. Add cilantro about 15 minutes before serving.
6. Leftovers can be frozen and reheated later.

pos,ingredient,amount,state,extra
0,onion,1 piece,cooked,no
1,garlic,2 clove,cooked,no
2,olive oil,2 tbsp,cooked,no
3,fresh cilantro,1 bunch,raw,no
4,tomatoes and green chilies,1 can,cooked,no
5,black beans,2 can,cooked,no
6,diced green chilies,1 can,cooked,no
7,oregano,1 tsp,cooked,no
8,cumin,1 tsp,cooked,no
9,jalapeno pepper,1 piece,cooked,yes


**Prep tasks** (4; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,onion,dice,,,True,False,1,Saute onion and garlic in olive oil.
2,garlic,mince,,,True,False,1,Saute onion and garlic in olive oil.
3,fresh cilantro,chop,,,False,False,5,Add cilantro about 15 minutes before serving.
4,jalapeno pepper,dice,,,True,False,2,"Add jalapeno pepper, if desired; this chile is spicy enough for most people without the jalapeno."


In [20]:
verdicts[8] = ["", "", "", ""]  # one per task row, in order: real | invented | wrong
missing[8] = ""  # prep the steps need that we did not return

In [21]:
card(9, order[8])

### 9. Mussels in Spicy Coconut Broth  
8 ingredients, 3 steps

**Steps**

1. Combine first 7 ingredients in a Dutch oven, and bring to a boil. Add mussels; cover and cook 5 minutes or until shells open.
2. Remove from heat; discard any unopened shells. Spoon 1 1/2 cups broth mixture into each of 2 soup bowls.
3. Divide mussels evenly among each soup bowl. Garnish with chopped basil.

pos,ingredient,amount,state,extra
0,light coconut milk,1 can,cooked,no
1,fresh ginger,10 g,cooked,no
2,sugar,1 tsp,cooked,no
3,lemon juice,1 tbsp,cooked,no
4,curry powder,1 tsp,cooked,no
5,salt,to taste,cooked,no
6,mussels,2 lb,cooked,no
7,fresh basil,1 bunch,raw,no


**Prep tasks** (2; the strict run found 1)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,fresh ginger,mince,,,True,False,1,"Combine first 7 ingredients in a Dutch oven, and bring to a boil. Add mussels; cover and cook 5 minutes or until shells open."
2,fresh basil,chop,,,False,True,3,Divide mussels evenly among each soup bowl. Garnish with chopped basil.


In [22]:
verdicts[9] = ["", ""]  # one per task row, in order: real | invented | wrong
missing[9] = ""  # prep the steps need that we did not return

In [23]:
card(10, order[9])

### 10. Cherokee Casserole  
11 ingredients, 7 steps

**Steps**

1. Cook rice per normal cooking instructions.
2. While rice cooks, brown meat and cook onions until tender.
3. Drain excess fat from meat.
4. Combine meat and onions in skillet, and add crushed tomatoes, mushroom soup, bay leaf, oregano, garlic powder, cumin, salt, pepper, olives.
5. Simmer together in skillet until rice is finished cooking.
6. Add rice, and mix all ingredients well.
7. Top casserole with cheese until melted.

pos,ingredient,amount,state,extra
0,long-grain rice,1 cup,cooked,no
1,tomatoes,1 can,cooked,no
2,ground beef,1 lb,cooked,no
3,onion,1 piece,cooked,no
4,bay leaf,1 piece,cooked,no
5,oregano,0.5 tsp,cooked,no
6,garlic powder,0.125 tsp,cooked,no
7,cumin,0.125 tsp,cooked,no
8,green olives,0.125 cup,cooked,no
9,cheddar cheese,12 oz,cooked,no


**Prep tasks** (1; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,onion,dice,,,True,False,2,"While rice cooks, brown meat and cook onions until tender."


In [24]:
verdicts[10] = [""]  # one per task row, in order: real | invented | wrong
missing[10] = ""  # prep the steps need that we did not return

In [25]:
card(11, order[10])

### 11. Acorn Squash, Pear and Azuki Soup With Sauteed Shiitakes  
10 ingredients, 6 steps

**Steps**

1. Preheat a large stock pot over medium heat. Add the oil and sauté the onions and peppers for about 10 minutes, or until onions just begin to brown.
2. Add the ginger and garlic and sauté a minute more. Add the salt, Chinese five spice, acorn squash and pear and cook for another minute, stirring often. Add the vegetable broth. Cover and bring to a boil. Once boiling, lower the heat just a bit to simmer briskly for about 20 more minutes, or until the squash is tender.
3. Puree half the soup using either an immersion blender or by transferring half the soup to a food processor or blender and then back into the rest of the soup (don’t forget to let the soup cool a bit so that the steam does not compress in the processor and kill you).
4. Add the azuki beans and lime. Cover and simmer on low, just until the beans are heated through. Meanwhile prepare the mushrooms.
5. Preheat a heavy bottomed skillet over medium high heat. Add the oils and sauté the mushrooms for about 7 minutes, until they are soft. Add the soy sauce and stir constantly until the soy sauce is absorbed (about 1 minute).
6. Ladle soup into bowls and top with sautéed mushrooms.

pos,ingredient,amount,state,extra
0,yellow onion,1 piece,cooked,no
1,garlic cloves,2 clove,cooked,no
2,salt,to taste,raw,no
3,Chinese five spice powder,1 tsp,cooked,no
4,acorn squash,2 piece,cooked,no
5,bartlett pears,2 piece,cooked,no
6,vegetable broth,1.5 l,cooked,no
7,azuki beans,1 cup,cooked,no
8,fresh lime juice,2 tbsp,raw,no
9,soy sauce,2 tbsp,cooked,no


**Prep tasks** (5; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,yellow onion,dice,,,True,False,1,"Preheat a large stock pot over medium heat. Add the oil and sauté the onions and peppers for about 10 minutes, or until onions just begin to brown."
2,garlic cloves,mince,,,True,False,2,"Add the ginger and garlic and sauté a minute more. Add the salt, Chinese five spice, acorn squash and pear and cook for another minute, stirring often. Add the vegetable broth. Cover and bring to a boil. Once boiling, lower the heat just a bit to simmer briskly for about 20 more minutes, or until the squash is tender."
3,acorn squash,dice,,,True,False,2,"Add the ginger and garlic and sauté a minute more. Add the salt, Chinese five spice, acorn squash and pear and cook for another minute, stirring often. Add the vegetable broth. Cover and bring to a boil. Once boiling, lower the heat just a bit to simmer briskly for about 20 more minutes, or until the squash is tender."
4,bartlett pears,dice,,,False,False,2,"Add the ginger and garlic and sauté a minute more. Add the salt, Chinese five spice, acorn squash and pear and cook for another minute, stirring often. Add the vegetable broth. Cover and bring to a boil. Once boiling, lower the heat just a bit to simmer briskly for about 20 more minutes, or until the squash is tender."
5,fresh ginger,mince,,,True,False,2,"Add the ginger and garlic and sauté a minute more. Add the salt, Chinese five spice, acorn squash and pear and cook for another minute, stirring often. Add the vegetable broth. Cover and bring to a boil. Once boiling, lower the heat just a bit to simmer briskly for about 20 more minutes, or until the squash is tender."


In [26]:
verdicts[11] = ["", "", "", "", ""]  # one per task row, in order: real | invented | wrong
missing[11] = ""  # prep the steps need that we did not return

In [27]:
card(12, order[11])

### 12. Linguine With Sun Dried Tomato Sauce  
6 ingredients, 4 steps

**Steps**

1. In large skillet, heat oil from tomatoes. Add garlic and cook over medium heat. Add canned tomatoes and sun-dried tomatoes and bring to a boil over medium heat.
2. Reduce heat and simmer, 5 minutes. Add cream and cook 5 minutes to blend flavors.
3. Meanwhile cook pasta. Drain.
4. Season sauce with pepper and stir in vinegar. Spoon over pasta and sprinkle with cheese.

pos,ingredient,amount,state,extra
0,sun-dried tomato packed in oil,0.75 cup,cooked,no
1,garlic cloves,3 clove,cooked,no
2,crushed tomatoes in puree,28 oz,cooked,no
3,heavy cream,0.33 cup,cooked,no
4,pepper,to taste,cooked,no
5,parmesan cheese,0.25 cup,raw,no
6,linguine,9 oz,cooked,yes
7,vinegar,1 tbsp,cooked,yes


**Prep tasks** (1; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,garlic cloves,mince,,,True,False,1,"In large skillet, heat oil from tomatoes. Add garlic and cook over medium heat. Add canned tomatoes and sun-dried tomatoes and bring to a boil over medium heat."


In [28]:
verdicts[12] = [""]  # one per task row, in order: real | invented | wrong
missing[12] = ""  # prep the steps need that we did not return

In [29]:
card(13, order[12])

### 13. Cheesy Scalloped Corn Bake  
11 ingredients, 6 steps

**Steps**

1. Set oven to 350 degrees.
2. Butter a 1-quart casserole dish.
3. In a bowl mix all ingredients together until well blended.
4. Transfer to prepared casserole dish.
5. Bake uncovered for 30-35 minutes.
6. Remove and top with shredded cheddar cheese (any amount desired) return to oven for 3-5 minutes or until cheese is melted.

pos,ingredient,amount,state,extra
0,eggs,2 piece,cooked,no
1,butter,2 tbsp,cooked,no
2,margarine,0.25 cup,cooked,no
3,evaporated milk,1 can,cooked,no
4,green bell pepper,1 piece,cooked,no
5,celery,1 piece,cooked,no
6,onion,1 piece,cooked,no
7,sugar,1 tbsp,cooked,no
8,seasoning salt,to taste,cooked,no
9,pepper,to taste,cooked,no


**Prep tasks** (3; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,green bell pepper,dice,,,True,False,3,In a bowl mix all ingredients together until well blended.
2,celery,dice,,,True,False,3,In a bowl mix all ingredients together until well blended.
3,onion,dice,,,True,False,3,In a bowl mix all ingredients together until well blended.


In [30]:
verdicts[13] = ["", "", ""]  # one per task row, in order: real | invented | wrong
missing[13] = ""  # prep the steps need that we did not return

In [31]:
card(14, order[13])

### 14. Barley Risotto With Caramelized Leeks and Mushrooms  
7 ingredients, 4 steps

**Steps**

1. Heat the oil in a large saucepan over medium heat. Add leek; cook 20 minutes or until tender and golden, stirring occasionally.
2. Add mushrooms; cook 5 minutes or until tender, stirring occasionally. Add barley, salt, and pepper; cook 1 minute, stirring constantly.
3. Add broth; bring to a boil. Cover, reduce heat, and simmer 30 minutes, stirring occasionally.
4. Uncover and cook 5 minutes or until liquid is absorbed. Stir in parsley.

pos,ingredient,amount,state,extra
0,olive oil,2 tbsp,cooked,no
1,leeks,3 piece,cooked,no
2,cremini mushrooms,8 oz,cooked,no
3,pearl barley,200 g,cooked,no
4,salt,to taste,cooked,no
5,fresh ground black pepper,to taste,cooked,no
6,fresh flat-leaf parsley,0.25 cup,raw,no
7,vegetable broth,6 cup,cooked,yes


**Prep tasks** (2; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,leeks,dice,,,True,False,1,"Heat the oil in a large saucepan over medium heat. Add leek; cook 20 minutes or until tender and golden, stirring occasionally."
2,fresh flat-leaf parsley,chop,,,False,False,4,Uncover and cook 5 minutes or until liquid is absorbed. Stir in parsley.


In [32]:
verdicts[14] = ["", ""]  # one per task row, in order: real | invented | wrong
missing[14] = ""  # prep the steps need that we did not return

In [33]:
card(15, order[14])

### 15. Better Tater Tot Casserole  
7 ingredients, 10 steps

**Steps**

1. Brown ground beef.
2. Add onions and garlic.
3. Fry until onions are translucent.
4. Add green beans, both soups, milk, and tater tots, reserving about 2 cups of tater tots for the top.
5. mix well.
6. place mixture in casserole dish.
7. place reserved tater tots on top, pushing in slightly.
8. sprinkle seasoning salt on top.
9. sprinkle grated cheese on top.
10. bake uncovored at 350 for 30 minutes.

pos,ingredient,amount,state,extra
0,green beans,2 can,cooked,no
1,onion,1 piece,cooked,no
2,garlic cloves,2 clove,cooked,no
3,seasoning salt,to taste,cooked,no
4,ground beef,1 lb,cooked,no
5,cheddar cheese,2 cup,cooked,no
6,milk,0.5 cup,cooked,no
7,cream of mushroom soup,1 can,cooked,yes
8,cream of chicken soup,1 can,cooked,yes
9,tater tots,2 lb,cooked,yes


**Prep tasks** (2; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,onion,dice,,,True,False,2,Add onions and garlic.
2,garlic cloves,mince,,,True,False,2,Add onions and garlic.


In [34]:
verdicts[15] = ["", ""]  # one per task row, in order: real | invented | wrong
missing[15] = ""  # prep the steps need that we did not return

In [35]:
card(16, order[15])

### 16. Black Pepper Pork with Ramen Noodles  
5 ingredients, 11 steps

**Steps**

1. Cut pork across grain into 1/4 inch wide strips; coat with mixture of garlic and 1 tbs. Stir-Fry Sauce.
2. Let stand 15 minutes.
3. Meanwhile, cook noodles only (not seasoning packet) in 2 qts boiling water 2 minutes; drain.
4. Heat 1 tbs oil in hot wok or large skillet over high heat.
5. Add pork and stir-fry 2 minutes; remove.
6. Reduce heat to medium-high; heat remaining vegetable oil in same pan.
7. Add onions; stir fry 1 minute.
8. Add broccoli and 2 tbs water; stir-fry 3 minutes.
9. Stir in pork, noodles, and remaining stir-fry sauce and pepper.
10. Cook, stirring well, until all ingredients are coated with sauce.
11. Add sesame oil and toss well to combine.

pos,ingredient,amount,state,extra
0,boneless pork chop,450 g,cooked,no
1,garlic,2 clove,cooked,no
2,onion,1 piece,cooked,no
3,broccoli florets,3 cup,cooked,no
4,black pepper,0.5 tsp,cooked,no
5,stir-fry sauce,3 tbsp,cooked,yes
6,vegetable oil,2 tbsp,cooked,yes
7,sesame oil,1 tsp,cooked,yes
8,ramen noodles,3 oz,cooked,yes


**Prep tasks** (4; the strict run found 2)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,boneless pork chop,slice,thin,,True,True,1,Cut pork across grain into 1/4 inch wide strips; coat with mixture of garlic and 1 tbs. Stir-Fry Sauce.
2,boneless pork chop,marinate,,garlic + stir-fry sauce,True,True,1,Cut pork across grain into 1/4 inch wide strips; coat with mixture of garlic and 1 tbs. Stir-Fry Sauce.
3,garlic,mince,,,True,False,1,Cut pork across grain into 1/4 inch wide strips; coat with mixture of garlic and 1 tbs. Stir-Fry Sauce.
4,onion,dice,,,True,False,7,Add onions; stir fry 1 minute.


In [36]:
verdicts[16] = ["", "", "", ""]  # one per task row, in order: real | invented | wrong
missing[16] = ""  # prep the steps need that we did not return

In [37]:
card(17, order[16])

### 17. Stir Fry Vegan Ramen  
10 ingredients, 5 steps

**Steps**

1. cooked instant noodles as package drain and set aside.
2. with sesame oil brown onion then add carrot saute for one minute then noodle.
3. add sauce and half cup of water and mix well then add beansprouts with spring onion for 2 minute.
4. if its too dry add all water then mix well til almost dry.
5. off heat top black pepper and mix well and serve.

pos,ingredient,amount,state,extra
0,water,0.5 cup,cooked,no
1,black pepper,to taste,cooked,no
2,bean sprouts,100 g,cooked,no
3,carrot,1 piece,cooked,no
4,spring onion,1 bunch,cooked,no
5,chili peppers,1 piece,raw,no
6,light soy sauce,3 tbsp,cooked,no
7,peri-peri,1 tsp,cooked,no
8,chili sauce,1 tbsp,cooked,no
9,dark soy sauce,2 tbsp,cooked,no


**Prep tasks** (3; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,onion,dice,,,True,False,2,with sesame oil brown onion then add carrot saute for one minute then noodle.
2,carrot,dice,,,True,False,2,with sesame oil brown onion then add carrot saute for one minute then noodle.
3,spring onion,chop,,,False,False,3,add sauce and half cup of water and mix well then add beansprouts with spring onion for 2 minute.


In [38]:
verdicts[17] = ["", "", ""]  # one per task row, in order: real | invented | wrong
missing[17] = ""  # prep the steps need that we did not return

In [39]:
card(18, order[17])

### 18. Smoked Salmon and Basil Rolls with Creme Fraiche or Sour Cream  
5 ingredients, 10 steps

**Steps**

1. Place 1 salmon slice on work surface.
2. Spread 2 teaspoons creme fraiche over salmon.
3. Sprinkle with 2 teaspoons chopped basil.
4. sprinkle with pepper.
5. Starting at 1 short end, roll up salmon slice, enclosing filling.
6. Spread 1 teaspoon creme fraiche over top of roll; sprinkle with 1 teaspoon chopped basil.
7. Cut roll crosswise into 5 slices (keep sliced roll together).
8. Repeat with remaining salmon, creme fraiche, and chopped basil.
9. (Can be made 6 hours ahead. Transfer salmon rolls to plate, cover and chill.) Arrange whole basil leaves on platter.
10. Using small spatula, top each basil leaf with 1 sliced salmon roll, fanning slightly.

pos,ingredient,amount,state,extra
0,smoked salmon,12 piece,raw,no
1,creme fraiche,0.75 cup,raw,no
2,sour cream,to taste,raw,no
3,fresh basil,0.75 cup,raw,no
4,fresh basil leaves,12 piece,raw,no


**Prep tasks** (1; the strict run found 1)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,fresh basil,chop,,,False,False,3,Sprinkle with 2 teaspoons chopped basil.


In [40]:
verdicts[18] = [""]  # one per task row, in order: real | invented | wrong
missing[18] = ""  # prep the steps need that we did not return

In [41]:
card(19, order[18])

### 19. Special Thai Chicken Salad  
20 ingredients, 8 steps

**Steps**

1. Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours).
2. If Using the Oven: Set oven to BROIL and place chicken on a broiling pan or a foil-lined baking sheet (turn out the sides of the foil to prevent juices from spilling into oven).
3. Place chicken on second-to-highest rung of your oven, allowing it to cook 5 minutes each side, then turning. Continue broiling and turning until cooked (chicken will be lightly charred around the edges). Use the leftover marinade to baste chicken the first time you turn it.
4. If Grilling the Chicken: Place chicken over a hot grill and cook until well done. Use leftover marinade to baste chicken when you turn it the first time.
5. While chicken is cooking, prepare the salad greens and dressing. To make the dressing: place water, lemongrass and lime leaf strips (or zest) in a sauce pan and bring to a boil. Boil for 1 minute (this will soften the lemongrass and bring out the flavor of the lime leaf). Remove from heat.
6. Add the rest of the dressing ingredients and mix well.
7. Taste-test the dressing. It should be tangy - a balance of salty, sweet, spicy, and sour. Adjust it to your taste by adding more fish sauce if not salty enough, more lime juice if too salty or sweet, more sugar if too sour or more chili sauce if not spicy enough for your taste.
8. Toss the salad with the dressing. Portion out the salad among individual plates, and top with strips of the hot chicken. Sprinkle with fresh basil and coriander; serve immediately while chicken is still warm.

pos,ingredient,amount,state,extra
0,boneless chicken breasts,3 piece,cooked,no
1,chicken thighs,2 piece,cooked,no
2,fresh basil,1 bunch,raw,no
3,fresh cilantro,1 bunch,raw,no
4,fish sauce,2 tbsp,raw,no
5,garlic cloves,3 clove,raw,no
6,soy sauce,1 tbsp,raw,no
7,lemons,1 piece,raw,no
8,lime juice,0.25 cup,raw,no
9,fresh ground black pepper,to taste,raw,no


**Prep tasks** (10; the strict run found 1)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,boneless chicken breasts,marinate,,fish sauce + garlic cloves + soy sauce + lime juice,True,True,1,"Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours)."
2,chicken thighs,marinate,,fish sauce + garlic cloves + soy sauce + lime juice,True,True,1,"Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours)."
3,boneless chicken breasts,rub,,fresh ground black pepper,True,True,1,"Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours)."
4,chicken thighs,rub,,fresh ground black pepper,True,True,1,"Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours)."
5,garlic cloves,mince,,,True,False,1,"Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours)."
6,garlic cloves,mince,,,True,False,6,Add the rest of the dressing ingredients and mix well.
7,kaffir lime leaf,slice,thin,,True,True,5,"While chicken is cooking, prepare the salad greens and dressing. To make the dressing: place water, lemongrass and lime leaf strips (or zest) in a sauce pan and bring to a boil. Boil for 1 minute (this will soften the lemongrass and bring out the flavor of the lime leaf). Remove from heat."
8,fresh basil,chop,,,False,False,8,"Toss the salad with the dressing. Portion out the salad among individual plates, and top with strips of the hot chicken. Sprinkle with fresh basil and coriander; serve immediately while chicken is still warm."
9,fresh cilantro,chop,,,False,False,8,"Toss the salad with the dressing. Portion out the salad among individual plates, and top with strips of the hot chicken. Sprinkle with fresh basil and coriander; serve immediately while chicken is still warm."
10,fresh cilantro,chop,,,False,False,8,"Toss the salad with the dressing. Portion out the salad among individual plates, and top with strips of the hot chicken. Sprinkle with fresh basil and coriander; serve immediately while chicken is still warm."


In [42]:
verdicts[19] = ["", "", "", "", "", "", "", "", "", ""]  # one per task row, in order: real | invented | wrong
missing[19] = ""  # prep the steps need that we did not return

In [43]:
card(20, order[19])

### 20. Avocado Lime Rice With Grilled Shrimp  
11 ingredients, 3 steps

**Steps**

1. Cook rice according to package directions. Meanwhile mix avocado with lime juice, cumin and half the salt and half the cilantro or parsley. Stir into rice.
2. Meanwhile, preheat grill to high, grease grate well. Toss shrimp with oil, garlic, lime zest and remaining salt. Thread evenly onto metal or soaked wooden skewers. Grill skewers, for 2 to 3 minutes per side or until shrimp start to curl and turn pink.
3. Serve rice with shrimp skewers. Garnish with remaining cilantro or parsley.

pos,ingredient,amount,state,extra
0,long grain white rice,1 cup,cooked,no
1,avocado,1 piece,raw,no
2,lime juice,2 tbsp,raw,no
3,ground cumin,1 tsp,cooked,no
4,salt,0.5 tsp,cooked,no
5,fresh cilantro,1 bunch,raw,no
6,parsley,1 bunch,raw,no
7,frozen shrimp,20 piece,cooked,no
8,olive oil,2 tbsp,cooked,no
9,garlic,2 clove,cooked,no


**Prep tasks** (3; the strict run found 0)

task,ingredient,technique,cut_size,with,make_ahead,stated,step,text of that step
1,fresh cilantro,chop,,,False,False,1,"Cook rice according to package directions. Meanwhile mix avocado with lime juice, cumin and half the salt and half the cilantro or parsley. Stir into rice."
2,parsley,chop,,,False,False,1,"Cook rice according to package directions. Meanwhile mix avocado with lime juice, cumin and half the salt and half the cilantro or parsley. Stir into rice."
3,garlic,mince,,,True,False,2,"Meanwhile, preheat grill to high, grease grate well. Toss shrimp with oil, garlic, lime zest and remaining salt. Thread evenly onto metal or soaked wooden skewers. Grill skewers, for 2 to 3 minutes per side or until shrimp start to curl and turn pink."


In [44]:
verdicts[20] = ["", "", ""]  # one per task row, in order: real | invented | wrong
missing[20] = ""  # prep the steps need that we did not return

## Shared prep across the 20 recipes

Tasks are grouped by (ingredient name, technique, form), the same key the graph will use. Names are only lower-cased and trimmed, with no real normalization yet (that is a separate roadmap task), so `garlic` and `garlic cloves` do not match here. A group in 2 or more recipes is prep we could do once.

In [45]:
records = []
for n, key in enumerate(order, 1):
    names = names_of(key)
    for t in current[key]["result"]["prep_tasks"]:
        records.append({
            "recipe": n,
            "recipe name": stage1[key]["name"],
            "ingredient": names[t["ingredient"]].lower().strip(),
            "technique": t["technique"],
            "form": form_of(t, names),
            "stated": t["stated"],
        })
flat = pd.DataFrame(records)

shared = (
    flat.groupby(["ingredient", "technique", "form"])
    .agg(recipes=("recipe", lambda s: sorted(set(s))), n=("recipe", "nunique"))
    .query("n >= 2")
    .sort_values("n", ascending=False)
)
display(Markdown(f"**Shared by 2 or more recipes: {len(shared)} groups**"))
display(shared)

near = (
    flat.groupby(["ingredient", "technique"])
    .agg(forms=("form", lambda s: sorted(set(s))), recipes=("recipe", lambda s: sorted(set(s))))
    .query("recipes.str.len() >= 2 and forms.str.len() >= 2")
)
display(Markdown(f"**Near-misses: same ingredient and technique, different form ({len(near)})**. These show where the rules may need tightening."))
display(near)

**Shared by 2 or more recipes: 7 groups**

,,,recipes,n
ingredient,technique,form,,
onion,dice,,"[2, 6, 8, 10, 13, 15, 16, 17]",8
garlic cloves,mince,,"[11, 12, 15, 19]",4
fresh basil,chop,,"[9, 18, 19]",3
fresh cilantro,chop,,"[8, 19, 20]",3
garlic,mince,,"[8, 16, 20]",3
fresh ginger,mince,,"[9, 11]",2
parsley,chop,,"[3, 20]",2


**Near-misses: same ingredient and technique, different form (0)**. These show where the rules may need tightening.

,,forms,recipes
ingredient,technique,,


## Your verdicts

In [46]:
out = []
for n, key in enumerate(order, 1):
    names = names_of(key)
    for k, t in enumerate(current[key]["result"]["prep_tasks"]):
        v = verdicts.get(n, [])
        out.append({
            "kind": "task", "n": n, "recipe": stage1[key]["name"], "task": k + 1,
            "ingredient": names[t["ingredient"]], "technique": t["technique"],
            "cut_size": t["cut_size"] or "", "with": " + ".join(names[p] for p in t["with"]),
            "stated": t["stated"], "source_step": t["source_step"],
            "verdict": v[k] if k < len(v) else "", "note": "",
        })
    if missing.get(n):
        out.append({"kind": "missing", "n": n, "recipe": stage1[key]["name"], "note": missing[n]})
review = pd.DataFrame(out)

if not REVIEWER:
    print("Set REVIEWER in the first code cell to save your review.")
else:
    review.to_csv(HERE / f"review_prep_{REVIEWER}.csv", index=False)
    filled = review[(review["kind"] == "task") & (review["verdict"] != "")]
    print(f"saved review_prep_{REVIEWER}.csv ({len(filled)} of {len(tasks)} task verdicts filled)")
    if len(filled):
        display(pd.crosstab(filled["stated"].map({True: "stated", False: "implied"}), filled["verdict"]))

Set REVIEWER in the first code cell to save your review.
